In [1]:
# Set up monitoring of changes to the module
using Revise
using Pkg

gpu_env = Base.expanduser("~/.julia/environments/MPSCircuitsGPU")
Pkg.activate(gpu_env)
#Pkg.activate(".")
println("Activated project: ", Base.active_project())

# Load module and others
using MPSCircuits, ITensors, ITensorMPS, LinearAlgebra, CUDA
#using MPSCircuits, ITensors, ITensorMPS, LinearAlgebra

  Activating project at `~/.julia/environments/MPSCircuitsGPU`


Activated project: /data/home/matthewg/.julia/environments/MPSCircuitsGPU/Project.toml



SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [8]:
Revise.revise()
Revise.errors()

In [2]:
# Benchmark-style chi=512 fixture used in backend comparison runs
using Random
using ITensors
using ITensorMPS

function build_fixture_mps_random(N::Int, chi_target::Int; seed::Int=1234)
    sites = siteinds("S=1/2", N)
    Random.seed!(seed + chi_target)
    mps = random_mps(sites; linkdims=chi_target)
    ITensorMPS.orthogonalize!(mps, 1)
    return mps
end

const BENCH_N = 24
const BENCH_CHI_TARGET = 512
const BENCH_SEED = 1234

mps_fixture = build_fixture_mps_random(BENCH_N, BENCH_CHI_TARGET; seed=BENCH_SEED)
mps_fixture_dense = ITensorMPS.dense(mps_fixture)
mps_fixture_gpu_fp32 = MPSCircuits.to_backend(mps_fixture_dense, MPSCircuits.BackendGPU; precision=:fp32)

println("Built benchmark fixture MPS")
println("N = ", BENCH_N, ", chi_target = ", BENCH_CHI_TARGET, ", actual maxlinkdim = ", ITensorMPS.maxlinkdim(mps_fixture_dense))
println("CPU backend maxlinkdim = ", ITensorMPS.maxlinkdim(mps_fixture_dense))
println("GPU fp32 fixture ready: ", typeof(mps_fixture_gpu_fp32))

Built benchmark fixture MPS
N = 24, chi_target = 512, actual maxlinkdim = 512
CPU backend maxlinkdim = 512
GPU fp32 fixture ready: MPS


# Big optimization

In [9]:
using Printf

"""
Create a notebook-friendly live progress callback for compiler hook events.
The callback prints stage updates and a live optimization status line.
"""
function make_notebook_live_view(; min_interval_s::Float64=0.2)
    state = Dict{Symbol,Any}(
        :run_start_t => time(),
        :last_print_t => 0.0,
        :current_layer => 0,
        :total_layers => 0,
        :run_step_done => 0,
        :run_step_total => 0,
        :layer_step_done => 0,
        :layer_step_total => 0,
    )

    function callback(event::Symbol, payload::NamedTuple)
        now_t = time()
        should_print = (now_t - state[:last_print_t]) >= min_interval_s

        if event == :run_start
            state[:run_start_t] = now_t
            println("Starting compile | protocol=", payload.protocol, " backend=", payload.backend, " precision=", payload.precision)
            println("Layers: ", payload.n_layers_max, " | Iter/layer: ", payload.n_iterations_per_layer)
            state[:last_print_t] = now_t
            return
        end

        if event == :layer_start
            state[:current_layer] = payload.layer
            state[:total_layers] = payload.total_layers
            println("Layer ", payload.layer, "/", payload.total_layers, " started | circuit_len=", payload.current_circuit_len, " | maxlinkdim(work)=", payload.mps_work_maxlinkdim)
            state[:last_print_t] = now_t
            return
        end

        if event == :layer_opt_start
            state[:run_step_total] = payload.run_opt_est_total_steps
            println("Layer ", payload.layer, " optimization started | layer steps=", payload.layer_opt_total_steps, " | run est total=", payload.run_opt_est_total_steps)
            state[:last_print_t] = now_t
            return
        end

        if event == :opt_step_done
            state[:run_step_done] = payload.run_step_done
            state[:run_step_total] = payload.run_step_est_total
            state[:layer_step_done] = payload.layer_step_done
            state[:layer_step_total] = payload.layer_step_total

            if should_print
                elapsed = now_t - state[:run_start_t]
                msg = @sprintf(
                    "\rLayer %d/%d | layer step %d/%d | run step %d/%d | ket χ=%d | bra χ=%d | elapsed %.1fs",
                    Int(state[:current_layer]),
                    Int(state[:total_layers]),
                    Int(state[:layer_step_done]),
                    Int(state[:layer_step_total]),
                    Int(state[:run_step_done]),
                    Int(max(1, state[:run_step_total])),
                    Int(payload.ket_maxlinkdim),
                    Int(payload.bra_maxlinkdim),
                    elapsed,
                )
                print(msg)
                flush(stdout)
                state[:last_print_t] = now_t
            end
            return
        end

        if event == :layer_done
            println()
            println("Layer ", payload.layer, " complete | circuit_len=", payload.circuit_len, " | layer_elapsed=", round(payload.layer_elapsed_s, digits=3), "s")
            state[:last_print_t] = now_t
            return
        end

        if event == :run_done
            println()
            println("Compile complete | layers=", payload.total_layers_done, " | opt steps=", payload.total_opt_steps_done, " | elapsed=", round(payload.total_elapsed_s, digits=3), "s")
            state[:last_print_t] = now_t
            return
        end
    end

    return callback
end

@assert isdefined(Main, :mps_fixture_dense) "Run Cell 3 first to build mps_fixture_dense."

live_view = make_notebook_live_view(; min_interval_s=0.15)
progress_tracker = MPSCircuits.CallbackProgressTracker(live_view; enabled=true, emit_every=1, include_env_norm=false)

circuit_demo = MPSCircuits.compile_mps_circuit(
    mps_fixture_dense,
    MPSCircuits.DecomposeAllAnalytical();
    n_layers_max=100,
    tolerance=1e-8,
    max_bond_dim=2 * ITensorMPS.maxlinkdim(mps_fixture_dense),
    working_cutoff=1e-12,
    backend=:gpu,
    gpu_fallback=true,
    precision=:fp32,
    transfer_policy=:strict_single_backend,
    progress=progress_tracker,
    
    )

circuit_demo_cpu = MPSCircuits.to_backend(circuit_demo, MPSCircuits.BackendCPU; precision=:preserve)
fidelity_demo = MPSCircuits.evaluate_circuit_fidelity(circuit_demo_cpu, mps_fixture_dense; cutoff=1e-12)
println("Final fidelity: ", fidelity_demo, " | gates: ", length(circuit_demo))

Starting compile | protocol=decompose_all backend=BackendGPU precision=fp32
Layers: 100 | Iter/layer: 0
Layer 1/100 started | circuit_len=0 | maxlinkdim(work)=512

Layer 1 complete | circuit_len=23 | layer_elapsed=1.074s
Layer 2/100 started | circuit_len=23 | maxlinkdim(work)=1024

Layer 2 complete | circuit_len=46 | layer_elapsed=1.906s
Layer 3/100 started | circuit_len=46 | maxlinkdim(work)=1024

Layer 3 complete | circuit_len=69 | layer_elapsed=2.363s
Layer 4/100 started | circuit_len=69 | maxlinkdim(work)=1024

Layer 4 complete | circuit_len=92 | layer_elapsed=1.865s
Layer 5/100 started | circuit_len=92 | maxlinkdim(work)=1024

Layer 5 complete | circuit_len=115 | layer_elapsed=1.879s
Layer 6/100 started | circuit_len=115 | maxlinkdim(work)=1024

Layer 6 complete | circuit_len=138 | layer_elapsed=1.842s
Layer 7/100 started | circuit_len=138 | maxlinkdim(work)=1024

Layer 7 complete | circuit_len=161 | layer_elapsed=1.845s
Layer 8/100 started | circuit_len=161 | maxlinkdim(work)=102

In [ ]:
circuit_demo = MPSCircuits.compile_mps_circuit(
    mps_fixture_dense,
    MPSCircuits.IterativeDecomposeOptimizeAll();
    optimization_protocol=MPSCircuits.TelescopingEnvironment(),
    n_layers_max=2,
    n_iterations_per_layer=1,
    tolerance=1e-8,
    max_bond_dim=2 * ITensorMPS.maxlinkdim(mps_fixture_dense),
    working_cutoff=1e-12,
    backend=:gpu,
    gpu_fallback=true,
    precision=:fp32,
    transfer_policy=:strict_single_backend,
    progress=progress_tracker,
    
    )

# Older tests

In [3]:
# 1. Setup Parameters
N = 50
j_coupling = 1.0
h_field = 0.5  # Transverse field

# 2. Define Site Indices
# "S=1/2" is the standard site type for qubits/spins
sites = siteinds("S=1/2", N)

# 3. Construct Hamiltonian using OpSum
os = OpSum()
for j in 1:(N - 1)
  # Interaction term: -J * Z_i * Z_{i+1}
  os += -j_coupling, "Sz", j, "Sz", j + 1
end
for j in 1:N
  # Field term: -h * X_i
  os += -h_field, "Sx", j
end

# Convert OpSum to MPO
H = MPO(os, sites)

# 4. Initialize State
# Start with a random product state (bond dimension 1)
# or a specific one like "Up"
state = [isodd(n) ? "Up" : "Dn" for n in 1:N]
psi_init = MPS(sites, state)

# 5. DMRG Settings (Sweeps)
# Each sweep gradually increases bond dimension (maxdim) 
# and decreases the truncation error (cutoff)
nsweeps = 5
maxdim = [10, 20, 100]
cutoff = [1e-10]

# 6. Run DMRG
energy, mps = dmrg(H, psi_init; nsweeps, maxdim, cutoff)

println("Ground State Energy: ", energy)

After sweep 1 energy=-15.806600464330774  maxlinkdim=4 maxerr=3.32E-16 time=15.073
After sweep 2 energy=-15.824805632828388  maxlinkdim=13 maxerr=9.95E-11 time=0.061
After sweep 3 energy=-15.825234680395129  maxlinkdim=18 maxerr=9.95E-11 time=0.080
After sweep 4 energy=-15.825290952266425  maxlinkdim=17 maxerr=9.88E-11 time=0.073
After sweep 5 energy=-15.825297037672172  maxlinkdim=16 maxerr=9.91E-11 time=0.070
Ground State Energy: -15.825297037672172


In [4]:
circuit = MPSCircuits.compile_mps_circuit(mps, MPSCircuits.DecomposeAllAnalytical(); n_layers_max=10, tolerance=1e-4)
MPSCircuits.evaluate_circuit_fidelity(circuit, mps; cutoff=1e-12)

0.9632741507382507

In [5]:
mpscu = cu(mps)

50-element MPS:
 ((dim=2|id=377|"Link,l=1"), (dim=2|id=953|"S=1/2,Site,n=1"))
 ((dim=4|id=509|"Link,l=2"), (dim=2|id=94|"S=1/2,Site,n=2"), (dim=2|id=377|"Link,l=1"))
 ((dim=2|id=972|"S=1/2,Site,n=3"), (dim=7|id=74|"Link,l=3"), (dim=4|id=509|"Link,l=2"))
 ((dim=2|id=819|"S=1/2,Site,n=4"), (dim=9|id=39|"Link,l=4"), (dim=7|id=74|"Link,l=3"))
 ((dim=2|id=940|"S=1/2,Site,n=5"), (dim=10|id=962|"Link,l=5"), (dim=9|id=39|"Link,l=4"))
 ((dim=2|id=839|"S=1/2,Site,n=6"), (dim=10|id=880|"Link,l=6"), (dim=10|id=962|"Link,l=5"))
 ((dim=2|id=528|"S=1/2,Site,n=7"), (dim=12|id=646|"Link,l=7"), (dim=10|id=880|"Link,l=6"))
 ((dim=2|id=19|"S=1/2,Site,n=8"), (dim=13|id=66|"Link,l=8"), (dim=12|id=646|"Link,l=7"))
 ((dim=2|id=149|"S=1/2,Site,n=9"), (dim=13|id=861|"Link,l=9"), (dim=13|id=66|"Link,l=8"))
 ((dim=2|id=395|"S=1/2,Site,n=10"), (dim=14|id=608|"Link,l=10"), (dim=13|id=861|"Link,l=9"))
 ⋮
 ((dim=2|id=990|"S=1/2,Site,n=42"), (dim=13|id=244|"Link,l=42"), (dim=12|id=753|"Link,l=41"))
 ((dim=2|id=889|"S=

In [15]:
ip_gpu = inner(mpscu,mpscu)

1.0000005f0

In [16]:
typeof(ip_gpu)

Float32

In [13]:
inner(mps,mps)

1.000000000000007

In [2]:
# 1. Setup Parameters
N = 50
j_coupling = 1.0
h_field = 0.5  # Transverse field

# 2. Define Site Indices
# "S=1/2" is the standard site type for qubits/spins
sites = siteinds("S=1/2", N)

# 3. Construct Hamiltonian using OpSum
os = OpSum()
for j in 1:(N - 1)
  # Interaction term: -J * Z_i * Z_{i+1}
  os += -j_coupling, "Sz", j, "Sz", j + 1
end
for j in 1:N
  # Field term: -h * X_i
  os += -h_field, "Sx", j
end

# Convert OpSum to MPO
H = MPO(os, sites)

# 4. Initialize State
# Start with a random product state (bond dimension 1)
# or a specific one like "Up"
state = [isodd(n) ? "Up" : "Dn" for n in 1:N]
psi_init = MPS(sites, state)

# --- NEW: MOVE TO GPU ---
# 5. Transfer the MPO and MPS to the GPU
H_gpu = cu(H) 
psi_init_gpu = cu(psi_init)
# ------------------------

# 5. DMRG Settings (Sweeps)
# Each sweep gradually increases bond dimension (maxdim) 
# and decreases the truncation error (cutoff)
nsweeps = 5
maxdim = [10, 20, 100]
cutoff = [1e-10]

# 6. Run DMRG
energy, mps = dmrg(H_gpu, psi_init_gpu; nsweeps, maxdim, cutoff)

println("Ground State Energy: ", energy)

After sweep 1 energy=-15.806599  maxlinkdim=4 maxerr=8.94E-08 time=24.341
After sweep 2 energy=-15.824641  maxlinkdim=14 maxerr=9.05E-08 time=0.799
After sweep 3 energy=-15.825252  maxlinkdim=22 maxerr=1.55E-08 time=0.515
After sweep 4 energy=-15.825266  maxlinkdim=22 maxerr=5.86E-09 time=0.621
After sweep 5 energy=-15.825249  maxlinkdim=17 maxerr=5.00E-09 time=0.430
Ground State Energy: -15.825249
